# phase1 ablation

In [1]:
import os
import re
import cv2
import json
import math
import numpy as np
import pandas as pd
import tempfile
import shutil
from tqdm import tqdm
from typing import Dict

import mediapipe as mp
# from feat import Detector  # 延迟加载，防止在消融实验时白白占用显存
import ollama

# ================= 核心配置区 =================
LLM_MODEL_NAME = "qwen2.5:7b" 

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results_ablation/remove_phase1_au2.jsonl" 

# =====================================================================
# 🎛️ ABLATION STUDY CONTROLS (消融实验控制面板)
# =====================================================================
ENABLE_PHASE1 = True       # 设为 False 则完全去除 Phase 1 (LLM将在无视觉证据下盲猜)
ENABLE_GAZE   = True       # 设为 False 则去除视线特征 (MediaPipe)
ENABLE_AU     = False       # 设为 False 则去除面部动作单元特征 (PyFeat)
# =====================================================================

TARGET_FRAME_COUNT = 10 

# --- Phase 1 阈值配置 ---
Z_SCORE_THRESH_MICRO = 2.0  
MIN_DURATION_SEC = 0.2      
ANALYSIS_FPS_LIMIT = 20.0  

# =====================================================================
# 模型按需加载 (节省显存)
# =====================================================================
face_mesh = None
detector = None

if ENABLE_PHASE1:
    if ENABLE_GAZE:
        print("🚀 初始化 Perception 基建 (MediaPipe - 用于 Gaze)...")
        mp_face_mesh = mp.solutions.face_mesh
        face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)
    
    if ENABLE_AU:
        print("🚀 初始化 Perception 基建 (PyFeat - 用于 AU)...")
        from feat import Detector
        detector = Detector(face_model="retinaface", landmark_model="mobilefacenet", au_model="svm", emotion_model="resmasknet", device="cuda")

# =====================================================================
# Phase 1: Identity-Calibrated Symbolic Parser
# =====================================================================

def extract_visual_signals_optimized(video_path: str) -> Dict[str, np.ndarray]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return {}
        
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if not original_fps or math.isnan(original_fps) or original_fps <= 0:
        original_fps = 25.0
        
    step = max(1, int(round(original_fps / ANALYSIS_FPS_LIMIT)))
    actual_fps = original_fps / step

    timestamps, gaze_y_list, gaze_x_list, frame_buffer = [], [], [],[]
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx % step == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            timestamps.append(frame_idx / original_fps)
            
            # [Ablation] 是否提取 Gaze 信号
            if ENABLE_GAZE and face_mesh:
                results = face_mesh.process(rgb_frame)
                if results.multi_face_landmarks:
                    lm = results.multi_face_landmarks[0].landmark
                    eye_h = abs(lm[145].y - lm[159].y) + 1e-6
                    eye_w = abs(lm[133].x - lm[33].x) + 1e-6
                    gaze_y_list.append((lm[145].y - lm[468].y) / eye_h)
                    gaze_x_list.append((lm[468].x - lm[33].x) / eye_w)
                else:
                    gaze_y_list.append(gaze_y_list[-1] if gaze_y_list else 0.5)
                    gaze_x_list.append(gaze_x_list[-1] if gaze_x_list else 0.5)
            else:
                # 填充占位符
                gaze_y_list.append(0.5)
                gaze_x_list.append(0.5)
            
            frame_buffer.append(frame) 

        frame_idx += 1
    cap.release()

    if not frame_buffer: return {}

    signals = {
        "fps": actual_fps, 
        "timestamps": np.array(timestamps),
        "gaze_y": np.array(gaze_y_list), 
        "gaze_x": np.array(gaze_x_list)
    }

    au_nums =["01", "02", "04", "06", "07", "12", "14", "17", "23", "24"]
    au_targets =[f"AU{au}" for au in au_nums]
    
    # [Ablation] 是否提取 AU 信号
    if ENABLE_AU and detector:
        temp_dir = tempfile.mkdtemp()
        temp_paths =[]
        try:
            for i, img in enumerate(frame_buffer):
                tmp_p = os.path.join(temp_dir, f"f_{i:03d}.jpg")
                cv2.imwrite(tmp_p, img) 
                temp_paths.append(tmp_p)
            
            fex = detector.detect_image(temp_paths, batch_size=len(temp_paths))
            df = fex.to_dataframe() if hasattr(fex, "to_dataframe") else pd.DataFrame(fex)
            
            for au in au_nums:
                target_key = f"AU{au}"
                col = next((c for c in[f"AU{au}_r", f"AU{au}", f"AU{au}_c", f"AU{int(au)}_r", f"AU{int(au)}", f"AU{int(au)}_c"] if c in df.columns), None)
                if col:
                    signals[target_key] = df[col].astype(float).fillna(0.0).values
                else:
                    signals[target_key] = np.zeros(len(frame_buffer))
                    
        except Exception as e:
            print(f"⚠️ PyFeat Failed: {e}")
            for t in au_targets: signals[t] = np.zeros(len(frame_buffer))
        finally:
            if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    else:
        # 填充全 0 占位符
        for t in au_targets: 
            signals[t] = np.zeros(len(frame_buffer))
            
    min_len = min(len(signals["timestamps"]), len(signals["gaze_y"]))
    for k in signals.keys():
        if k != "fps" and isinstance(signals[k], np.ndarray):
            signals[k] = signals[k][:min_len]

    return signals

def z_score_symbolic_parser(signals: Dict) -> str:
    if not signals or "timestamps" not in signals: return "[Error] No signals."
    timestamps = signals["timestamps"]
    events =[]

    track_dict = {}
    if ENABLE_GAZE:
        track_dict.update({
            "gaze_y": {"name": "EYE-ROLL (Upward Gaze)", "type": "positive"},
            "gaze_x": {"name": "SIDE-EYE (Lateral Gaze)", "type": "absolute"},
        })
    if ENABLE_AU:
        track_dict.update({
            "AU14": {"name": "SMIRK_CONTEMPT (Unilateral Lip Tightener)", "type": "positive"},
            "AU24": {"name": "LIP_PRESS (Suppressed Emotion)", "type": "positive"},
            "AU23": {"name": "LIP_TIGHTEN (Frustration)", "type": "positive"},
            "AU07": {"name": "SQUINT (Suspicion/Disbelief)", "type": "positive"},
            "AU04": {"name": "BROW_FURROW (Disapproval)", "type": "positive"},
            "AU01": {"name": "BROW_RAISE (Surprise/Disbelief)", "type": "positive"},
        })

    for key, meta in track_dict.items():
        if key not in signals: continue
        arr = signals[key]
        if len(arr) == 0: continue
        
        p25, p75 = np.percentile(arr, 25), np.percentile(arr, 75)
        neutral_mask = (arr >= p25) & (arr <= p75)
        mu = np.mean(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.mean(arr)
        sigma = np.std(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.std(arr)
        sigma = max(sigma, 0.05) 
        
        z_scores = (arr - mu) / sigma
        
        if meta["type"] == "positive": anomalies = z_scores > Z_SCORE_THRESH_MICRO
        else: anomalies = np.abs(z_scores) > Z_SCORE_THRESH_MICRO

        in_event = False
        start_idx = 0
        for i, is_abnormal in enumerate(anomalies):
            if is_abnormal and not in_event:
                in_event, start_idx = True, i
            elif not is_abnormal and in_event:
                in_event = False
                dur = timestamps[i-1] - timestamps[start_idx]
                if dur >= MIN_DURATION_SEC:
                    peak_z = min(np.max(np.abs(z_scores[start_idx:i])), 15.0)
                    events.append({
                        "t_start": timestamps[start_idx],
                        "t_end": timestamps[i-1],
                        "event": meta["name"],
                        "intensity": round(float(peak_z), 1)
                    })

    # [Ablation] Deadpan 逻辑仅在 AU 开启时运算
    if ENABLE_AU:
        deadpan_cols =["AU12", "AU14", "AU23", "AU24", "AU04", "AU01", "AU02", "AU07"]
        deadpan_stack = np.vstack([signals[c] for c in deadpan_cols if c in signals])
        # 确保数据不全是0占位符（避免错误触发持续的DEADPAN）
        if deadpan_stack.size and np.any(deadpan_stack):
            mask_deadpan = np.all(deadpan_stack < 1.0, axis=0) 
            in_event, start_idx = False, 0
            for i, is_dp in enumerate(mask_deadpan):
                if is_dp and not in_event:
                    in_event, start_idx = True, i
                elif not is_dp and in_event:
                    in_event = False
                    if timestamps[i-1] - timestamps[start_idx] >= 0.5:
                        events.append({"t_start": timestamps[start_idx], "t_end": timestamps[i-1], "event": "DEADPAN (Emotionless)", "intensity": 5.0})

    events = sorted(events, key=lambda x: x["t_start"])
    
    report = "[DETECTED VISUAL EVENTS (Z-Score Calibrated)]:\n"
    if not events: report += "- Normal / Neutral Gaze and Expressions.\n"
    else:
        for e in events: report += f"- At {e['t_start']:.1f}s - {e['t_end']:.1f}s: {e['event']} (Intensity: {e['intensity']})\n"
    return report

# =====================================================================
# Phase 2: ToM Reasoning 
# =====================================================================

OLLAMA_OPTIONS = {'temperature': 0.1}

def phase2_tom_draft(symbolic_report: str) -> str:
    prompt = f"""
You are a behavioral psychology expert. Recent cognitive research indicates that human sarcasm detection is a holistic, non-sequential process rather than a step-by-step logical deduction.

### VISUAL EVIDENCE (Tensor of Cues):
{symbolic_report}

### HOLISTIC INTEGRATION GUIDELINES:
Do NOT think step-by-step. Instead, treat the visual evidence as a "Bag of Cues" and synthesize them instantly across three independent, parallel dimensions:
- Dimension 1 (Eye Dynamics): Look for high-intensity Gaze anomalies (e.g., EYE-ROLL, SIDE-EYE).
- Dimension 2 (Lower Face Dynamics): Look for suppressed or asymmetric mouth movements (e.g., SMIRK, LIP_PRESS, LIP_TIGHTEN, DEADPAN).
- Dimension 3 (Upper Face Dynamics): Look for cognitive dissonance markers (e.g., SQUINT, BROW_FURROW, BROW_RAISE).

[Sarcasm Synthesis Rule]: 
Sarcasm is highly probable if there is a strong anomaly (high Z-score intensity) in ANY of these orthogonal dimensions, or a synergistic combination of them (e.g., an eye-roll coupled with a smirk). If the evidence shows "Normal / Neutral", it lacks the necessary leakage for sarcasm.

### OUTPUT FORMAT:
Provide your response in this exact format:
Reasoning:[Write 1-2 sentences holistically synthesizing the dimensions to state if sufficient sarcastic cues are present]
Final_JSON: {{"sarcasm": true}} or {{"sarcasm": false}}
    """

    try:
        response = ollama.chat(model=LLM_MODEL_NAME, messages=[{'role': 'user', 'content': prompt}], options=OLLAMA_OPTIONS)
        content = response['message']['content'].strip()
        if not content:
            return "Reasoning: Empty response. \nFinal_JSON: {\"sarcasm\": false}"
        return content
    except Exception as e:
        return f"Reasoning: Error {e}. \nFinal_JSON: " + "{\"sarcasm\": false}"

# ================= 工具函数 =================

def clean_and_parse_json(content: str) -> Dict:
    content = str(content).strip()
    is_sarcastic = False
    if re.search(r'"sarcasm"\s*:\s*true', content, re.IGNORECASE):
        is_sarcastic = True
        
    reasoning = ""
    m = re.search(r'Reasoning:\s*(.*?)(?=Final_JSON|$)', content, re.DOTALL | re.IGNORECASE)
    if m: reasoning = m.group(1).strip()
        
    return {"Is_Sarcastic": is_sarcastic, "reasoning": reasoning}

# ================= 主流程 =================
def main():
    if not os.path.exists(os.path.dirname(OUTPUT_FILE)):
        os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(INPUT_DATA_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try: processed_ids.add(json.loads(line)['video_id'])
                except: pass

    # 打印当前实验状态
    print("\n" + "="*50)
    print(f"🚀 开始处理 Ablation Pipeline")
    print(f"   - Phase 1 Enabled: {ENABLE_PHASE1}")
    print(f"   - Gaze Enabled:    {ENABLE_GAZE}")
    print(f"   - AU Enabled:      {ENABLE_AU}")
    print("="*50 + "\n")

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
        items = list(data.items()) if isinstance(data, dict) else list(enumerate(data))
        
        for vid_id, item in tqdm(items, desc="Reasoning"):
            if isinstance(data, list): vid_id = item.get('video_id', str(vid_id))
            if vid_id in processed_ids: continue

            video_path = os.path.join(VIDEO_FOLDER, f"{vid_id}.mp4")
            if not os.path.exists(video_path): continue

            try:
                # [Ablation] 完全去除 Phase1
                if not ENABLE_PHASE1:
                    symbolic_report = "No visual evidence provided (Phase 1 Ablated)."
                else:
                    signals = extract_visual_signals_optimized(video_path)
                    if not signals: continue 
                    symbolic_report = z_score_symbolic_parser(signals)
                
                # 2. ToM 逻辑推理
                current_draft = phase2_tom_draft(symbolic_report)
                final_json = clean_and_parse_json(current_draft)
                
                # 结果写入
                result_entry = {
                    "video_id": vid_id,
                    "ground_truth": item.get("sarcasm", None),
                    "prediction": final_json.get("Is_Sarcastic", False),
                    "neuro_symbolic_trace": {
                        "symbolic_report": symbolic_report, 
                        "final_reasoning": final_json.get("reasoning", ""),
                        "raw_output": current_draft
                    }
                }
                
                out_f.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
                out_f.flush()
                
            except Exception as e:
                print(f"❌ Error processing {vid_id}: {e}")
                err_entry = {"video_id": vid_id, "error": str(e)}
                out_f.write(json.dumps(err_entry) + "\n")
                out_f.flush()

    print("✅ 消融实验处理完毕。结果已保存至:", OUTPUT_FILE)

if __name__ == "__main__":
    main()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


🚀 初始化 Perception 基建 (MediaPipe - 用于 Gaze)...

🚀 开始处理 Ablation Pipeline
   - Phase 1 Enabled: True
   - Gaze Enabled:    True
   - AU Enabled:      False



Reasoning: 100%|██████████| 690/690 [24:46<00:00,  2.15s/it]

✅ 消融实验处理完毕。结果已保存至: ./results_ablation/remove_phase1_au2.jsonl


# phase 2

In [ ]:
import os
import cv2
import csv
import json
import math
import shutil
import tempfile
import numpy as np
import pandas as pd
import re
from typing import Dict, List
from tqdm import tqdm
from dataclasses import dataclass
import mediapipe as mp
import ollama

# 假设你使用的特征提取库
from feat import Detector 

LLM_MODEL_NAME = "qwen2.5:7b" 
VIDEO_FOLDER = "./final_utterance_videos"  
INPUT_DATA_FILE = "./mustardpp.csv"

# ================= 审稿人视角的科学配置：消融实验控制面板 =================
@dataclass
class AblationConfig:
    # 实验开关，修改此处即可跑不同版本的实验
    use_z_score_calibration: bool = True  # True: 基于P25-P75基线; False: 退化为全局标准化
    use_semantic_dict: bool = True        # True: AU14->SMIRK; False: AU14->AU14_Activation
    use_temporal_filter: bool = True      # True: 强制 MIN_DURATION_SEC; False: 去掉时长过滤，允许瞬态噪声

# 实例化当前实验配置
config = AblationConfig(
    use_z_score_calibration=True,
    use_semantic_dict=True,
    use_temporal_filter=True
)

# 动态生成结果文件名，避免实验结果互相覆盖 (严谨性体现)
exp_name = f"zscore_{config.use_z_score_calibration}_sem_{config.use_semantic_dict}_temp_{config.use_temporal_filter}"
OUTPUT_FILE = f"./results/ablation_{exp_name}.jsonl"
# 注意：缓存文件现在只存【原始视觉信号】，不存Report文本，这样无论怎么消融，都不需要重新跑提取！
RAW_SIGNALS_CACHE_FILE = "./results/mustardpp_raw_signals_cache.json"  

TARGET_FRAME_COUNT = 10 
Z_SCORE_THRESH_MICRO = 2.0  
MIN_DURATION_SEC = 0.2      
ANALYSIS_FPS_LIMIT = 20.0  

face_mesh = None

# ================= Phase 1 (保持不变，但增加 Numpy 序列化支持) =================
def extract_visual_signals_optimized(video_path: str, detector: 'Detector') -> Dict[str, np.ndarray]:
    global face_mesh
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return {}
        
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if not original_fps or math.isnan(original_fps) or original_fps <= 0:
        original_fps = 25.0
        
    step = max(1, int(round(original_fps / ANALYSIS_FPS_LIMIT)))
    actual_fps = original_fps / step

    timestamps, gaze_y_list, gaze_x_list, frame_buffer = [],[], [],[]
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx % step == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            timestamps.append(frame_idx / original_fps)
            
            results = face_mesh.process(rgb_frame)
            if results.multi_face_landmarks:
                lm = results.multi_face_landmarks[0].landmark
                eye_h = abs(lm[145].y - lm[159].y) + 1e-6
                eye_w = abs(lm[133].x - lm[33].x) + 1e-6
                gaze_y_list.append((lm[145].y - lm[468].y) / eye_h)
                gaze_x_list.append((lm[468].x - lm[33].x) / eye_w)
            else:
                gaze_y_list.append(gaze_y_list[-1] if gaze_y_list else 0.5)
                gaze_x_list.append(gaze_x_list[-1] if gaze_x_list else 0.5)
            
            frame_buffer.append(frame) 

        frame_idx += 1
    cap.release()

    if not frame_buffer: return {}

    signals = {
        "fps": actual_fps, "timestamps": np.array(timestamps),
        "gaze_y": np.array(gaze_y_list), "gaze_x": np.array(gaze_x_list)
    }

    au_nums =["01", "02", "04", "06", "07", "12", "14", "17", "23", "24"]
    au_targets =[f"AU{au}" for au in au_nums]
    
    temp_dir = tempfile.mkdtemp()
    temp_paths =[]
    
    try:
        for i, img in enumerate(frame_buffer):
            tmp_p = os.path.join(temp_dir, f"f_{i:03d}.jpg")
            cv2.imwrite(tmp_p, img) 
            temp_paths.append(tmp_p)
        
        fex = detector.detect_image(temp_paths, batch_size=8)
        df = fex.to_dataframe() if hasattr(fex, "to_dataframe") else pd.DataFrame(fex)
        
        for au in au_nums:
            target_key = f"AU{au}"
            col = next((c for c in[f"AU{au}_r", f"AU{au}", f"AU{au}_c", f"AU{int(au)}_r", f"AU{int(au)}", f"AU{int(au)}_c"] if c in df.columns), None)
            signals[target_key] = df[col].astype(float).fillna(0.0).values if col else np.zeros(len(frame_buffer))
                
    except Exception as e:
        print(f"⚠️ PyFeat Failed: {e}")
        for t in au_targets: signals[t] = np.zeros(len(frame_buffer))
    finally:
        if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
            
    min_len = min(len(signals["timestamps"]), len(signals["gaze_y"]))
    for k in signals.keys():
        if k != "fps" and isinstance(signals[k], np.ndarray):
            signals[k] = signals[k][:min_len]

    return signals

# ================= Phase 2 (带有严谨消融逻辑的 Symbolic Parser) =================
def z_score_symbolic_parser(signals: Dict, config: AblationConfig) -> str:
    if not signals or "timestamps" not in signals: return "[Error] No signals."
    timestamps = signals["timestamps"]
    events =[]

    track_dict = {
        "gaze_y": {"name": "EYE-ROLL (Upward Gaze)", "type": "positive"},
        "gaze_x": {"name": "SIDE-EYE (Lateral Gaze)", "type": "absolute"},
        "AU14": {"name": "SMIRK_CONTEMPT (Unilateral Lip Tightener)", "type": "positive"},
        "AU24": {"name": "LIP_PRESS (Suppressed Emotion)", "type": "positive"},
        "AU23": {"name": "LIP_TIGHTEN (Frustration)", "type": "positive"},
        "AU07": {"name": "SQUINT (Suspicion/Disbelief)", "type": "positive"},
        "AU04": {"name": "BROW_FURROW (Disapproval)", "type": "positive"},
        "AU01": {"name": "BROW_RAISE (Surprise/Disbelief)", "type": "positive"},
    }

    for key, meta in track_dict.items():
        if key not in signals: continue
        arr = np.array(signals[key]) # 确保为numpy array
        if len(arr) == 0: continue
        
        # --- Ablation 1: Z-Score Calibration (局部微表情基线 vs 全局均值) ---
        if config.use_z_score_calibration:
            p25, p75 = np.percentile(arr, 25), np.percentile(arr, 75)
            neutral_mask = (arr >= p25) & (arr <= p75)
            mu = np.mean(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.mean(arr)
            sigma = np.std(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.std(arr)
        else:
            # 消融：退化为最原始的全局均值方差，失去了对视频主体中性表情的定位能力
            mu = np.mean(arr)
            sigma = np.std(arr)
            
        sigma = max(sigma, 0.05) 
        z_scores = (arr - mu) / sigma
        
        if meta["type"] == "positive": anomalies = z_scores > Z_SCORE_THRESH_MICRO
        else: anomalies = np.abs(z_scores) > Z_SCORE_THRESH_MICRO

        # --- Ablation 2: Semantic Dictionary (高阶语义 vs 原始通道名) ---
        event_label = meta["name"] if config.use_semantic_dict else f"{key}_Activation"

        in_event = False
        start_idx = 0
        for i, is_abnormal in enumerate(anomalies):
            if is_abnormal and not in_event:
                in_event, start_idx = True, i
            elif not is_abnormal and in_event:
                in_event = False
                dur = timestamps[i-1] - timestamps[start_idx]
                
                # --- Ablation 3: Temporal Filtering (时域滤波平滑去噪 vs 允许瞬态噪声) ---
                passes_time_filter = (dur >= MIN_DURATION_SEC) if config.use_temporal_filter else True
                
                if passes_time_filter:
                    peak_z = min(np.max(np.abs(z_scores[start_idx:i])), 15.0)
                    events.append({
                        "t_start": timestamps[start_idx],
                        "t_end": timestamps[i-1],
                        "event": event_label,
                        "intensity": round(float(peak_z), 1)
                    })

    # Deadpan 逻辑同理，适配字典消融
    deadpan_cols =["AU12", "AU14", "AU23", "AU24", "AU04", "AU01", "AU02", "AU07"]
    deadpan_stack = np.vstack([signals[c] for c in deadpan_cols if c in signals])
    if deadpan_stack.size:
        mask_deadpan = np.all(deadpan_stack < 1.0, axis=0) 
        in_event, start_idx = False, 0
        for i, is_dp in enumerate(mask_deadpan):
            if is_dp and not in_event:
                in_event, start_idx = True, i
            elif not is_dp and in_event:
                in_event = False
                dur = timestamps[i-1] - timestamps[start_idx]
                passes_time_filter = (dur >= 0.5) if config.use_temporal_filter else True
                if passes_time_filter:
                    dp_label = "DEADPAN (Emotionless)" if config.use_semantic_dict else "ALL_AU_INACTIVE"
                    events.append({"t_start": timestamps[start_idx], "t_end": timestamps[i-1], "event": dp_label, "intensity": 5.0})

    events = sorted(events, key=lambda x: x["t_start"])
    
    report = f"[DETECTED VISUAL EVENTS ({'Z-Score Calibrated' if config.use_z_score_calibration else 'Global Normalized'})]:\n"
    if not events: report += "- Normal / Neutral Gaze and Expressions.\n"
    else:
        for e in events: report += f"- At {e['t_start']:.1f}s - {e['t_end']:.1f}s: {e['event']} (Intensity: {e['intensity']})\n"
    return report

# ================= Phase 3 =================
def phase2_tom_draft(symbolic_report: str, config: AblationConfig) -> str:
    # 动态构建Prompt：如果关闭了语义词典，我们要公平地去掉Prompt里关于高阶语义的暗示，考察LLM纯看AU的表现
    if config.use_semantic_dict:
        guideline_text = """
- Dimension 1 (Eye Dynamics): Look for high-intensity Gaze anomalies (e.g., EYE-ROLL, SIDE-EYE).
- Dimension 2 (Lower Face Dynamics): Look for suppressed or asymmetric mouth movements (e.g., SMIRK, LIP_PRESS, LIP_TIGHTEN, DEADPAN).
- Dimension 3 (Upper Face Dynamics): Look for cognitive dissonance markers (e.g., SQUINT, BROW_FURROW, BROW_RAISE)."""
    else:
         guideline_text = """
- Dimension 1 (Eye Dynamics): Look for high-intensity Gaze (y or x) anomalies.
- Dimension 2 (Lower Face Dynamics): Look for suppressed or asymmetric mouth movements (e.g., AU14, AU24, AU23, or lack of activations).
- Dimension 3 (Upper Face Dynamics): Look for cognitive dissonance markers (e.g., AU07, AU04, AU01)."""

    prompt = f"""
You are a behavioral psychology expert. Recent cognitive research indicates that human sarcasm detection is a holistic, non-sequential process rather than a step-by-step logical deduction.

# ### VISUAL EVIDENCE (Tensor of Cues):
# {symbolic_report}

# ### HOLISTIC INTEGRATION GUIDELINES:
# Do NOT think step-by-step. Instead, treat the visual evidence as a "Bag of Cues" and synthesize them instantly across three independent, parallel dimensions:{guideline_text}

[Sarcasm Synthesis Rule]: 
Sarcasm is highly probable if there is a strong anomaly (high Z-score intensity) in ANY of these orthogonal dimensions, or a synergistic combination of them. If the evidence shows "Normal / Neutral", it lacks the necessary leakage for sarcasm.

### OUTPUT FORMAT:
Provide your response in this exact format:
Reasoning:[Write 1-2 sentences holistically synthesizing the dimensions to state if sufficient sarcastic cues are present]
Final_JSON: {{"sarcasm": true}} or {{"sarcasm": false}}
    """

    try:
        response = ollama.chat(
            model=LLM_MODEL_NAME, 
            messages=[{'role': 'user', 'content': prompt}], 
            options={'temperature': 0.1}
        )
        content = response['message']['content'].strip()
        if not content: return "Reasoning: Empty response. \nFinal_JSON: {\"sarcasm\": false}"
        return content
    except Exception as e:
        return f"Reasoning: Error {e}. \nFinal_JSON: {{\"sarcasm\": false}}"

def clean_and_parse_json(content: str) -> Dict:
    content = str(content).strip()
    is_sarcastic = bool(re.search(r'"sarcasm"\s*:\s*true', content, re.IGNORECASE))
    reasoning = ""
    m = re.search(r'Reasoning:\s*(.*?)(?=Final_JSON|$)', content, re.DOTALL | re.IGNORECASE)
    if m: reasoning = m.group(1).strip()
    return {"Is_Sarcastic": is_sarcastic, "reasoning": reasoning}

# ================= 辅助函数：安全缓存Numpy信号 =================
def serialize_signals(signals):
    """将Numpy数组转为List，以便存入JSON"""
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in signals.items()}

def deserialize_signals(data):
    """将JSON读取的List转回Numpy数组"""
    return {k: (np.array(v) if isinstance(v, list) else v) for k, v in data.items()}

# ================= 主流程 =================
def main():
    global face_mesh
    
    if not os.path.exists(os.path.dirname(OUTPUT_FILE)):
        os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    print(f"==================================================")
    print(f"⚙️ 当前消融实验配置 [{exp_name}] :")
    print(f" - Z-Score Calibration: {'✅ ON' if config.use_z_score_calibration else '❌ OFF (Global Norm)'}")
    print(f" - Semantic Dictionary: {'✅ ON' if config.use_semantic_dict else '❌ OFF (Raw Feature Names)'}")
    print(f" - Temporal Filtering:  {'✅ ON' if config.use_temporal_filter else '❌ OFF (Allow Noise)'}")
    print(f"==================================================")

    data_items =[]
    with open(INPUT_DATA_FILE, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = row['KEY']
            if key.endswith('_u') and row['Sarcasm'].strip() in ['0', '1']:
                data_items.append({"video_id": key, "sarcasm": True if row['Sarcasm'].strip() == '1' else False})
                
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try: processed_ids.add(json.loads(line)['video_id'])
                except: pass
                
    # ================= 关键修改：只加载/缓存 Raw Signals，实现无损快速消融 =================
    raw_signals_cache = {}
    if os.path.exists(RAW_SIGNALS_CACHE_FILE):
        with open(RAW_SIGNALS_CACHE_FILE, 'r', encoding='utf-8') as f:
            try:
                raw_data = json.load(f)
                raw_signals_cache = {k: deserialize_signals(v) for k, v in raw_data.items()}
                print(f"📦 已加载本地【底层视觉信号】缓存: 包含 {len(raw_signals_cache)} 条记录。消融推断将极速进行！")
            except Exception as e:
                print(f"⚠️ 缓存文件读取失败: {e}，将重新建立。")
                
    need_visual_processing = any(item['video_id'] not in raw_signals_cache for item in data_items)
            
    detector = None
    if need_visual_processing:
        print("🚀 初始化 Perception 基建 (MediaPipe & PyFeat) - 补充缺失缓存...")
        mp_face_mesh = mp.solutions.face_mesh
        face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)
        detector = Detector(face_model="retinaface", landmark_model="mobilefacenet", au_model="svm", emotion_model="resmasknet", device="cuda")
    else:
        print("⚡ 待处理视频均已存在信号缓存，跳过加载重量级视觉模型，开始纯享LLM推断！")

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
        for item in tqdm(data_items, desc="Ablation Reasoning"):
            vid_id = item['video_id']
            ground_truth_label = item['sarcasm']
            
            if vid_id in processed_ids: 
                continue

            try:
                # 1. 提取或读取原始信号 (Phase 1)
                if vid_id in raw_signals_cache:
                    signals = raw_signals_cache[vid_id]
                else:
                    video_path = os.path.join(VIDEO_FOLDER, f"{vid_id}.mp4")
                    if not os.path.exists(video_path): continue
                        
                    signals = extract_visual_signals_optimized(video_path, detector)
                    if not signals: continue 
                    
                    # 缓存纯Numpy信号而不是文本，保证消融重跑的效率
                    raw_signals_cache[vid_id] = signals
                    with open(RAW_SIGNALS_CACHE_FILE, 'w', encoding='utf-8') as cf:
                        json.dump({k: serialize_signals(v) for k, v in raw_signals_cache.items()}, cf, ensure_ascii=False)
                
                # 2. 生成 Symbolic Report (根据当前配置动态消融)
                symbolic_report = z_score_symbolic_parser(signals, config)
                
                # 3. ToM 逻辑推理 
                current_draft = phase2_tom_draft(symbolic_report, config)
                final_json = clean_and_parse_json(current_draft)
                
                result_entry = {
                    "video_id": vid_id,
                    "ground_truth": ground_truth_label,
                    "prediction": final_json.get("Is_Sarcastic", False),
                    "ablation_settings": {
                        "z_score": config.use_z_score_calibration,
                        "semantic_dict": config.use_semantic_dict,
                        "temporal_filter": config.use_temporal_filter
                    },
                    "neuro_symbolic_trace": {
                        "symbolic_report": symbolic_report, 
                        "final_reasoning": final_json.get("reasoning", ""),
                        "critique_history": ["[ABLATION] Phase 3 removed."],
                    }
                }
                
                out_f.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
                out_f.flush()
                
            except Exception as e:
                print(f"❌ Error processing {vid_id}: {e}")

    print(f"✅ 消融实验处理完毕。结果已保存至: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

⚙️ 当前消融实验配置 [zscore_True_sem_True_temp_True] :
 - Z-Score Calibration: ✅ ON
 - Semantic Dictionary: ✅ ON
 - Temporal Filtering:  ✅ ON
⚠️ 缓存文件读取失败: 'str' object has no attribute 'items'，将重新建立。
🚀 初始化 Perception 基建 (MediaPipe & PyFeat) - 补充缺失缓存...


Reasoning: 100%|██████████| 690/690 [00:00<00:00, 303043.95it/s]

✅ 消融实验处理完毕。结果已保存至: ./results_ablation/ablation_zscore_True_sem_True_temp_True.jsonl


# result evaluation

In [2]:
import os
import json

# INPUT_FILE = "./results/ablation_gpt_vl.jsonl" 
INPUT_FILE = "./results_ablation/remove_phase1_au2.jsonl"  


def evaluate_and_extract_errors():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    # 初始化计数器
    stats = {
        "total": 0,
        "correct": 0,
        "tp": 0, # 真阳性 (GT=True, Pred=True)
        "tn": 0, # 真阴性 (GT=False, Pred=False)
        "fp": 0, # 假阳性 (GT=False, Pred=True) -> 误报
        "fn": 0  # 假阴性 (GT=True, Pred=False) -> 漏报
    }
    
    error_cases = []

    print(f"📂 正在读取并修复文件格式: {INPUT_FILE} ...")

    # 1. 读取整个文件内容到内存
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        content = f.read()

    # 2. 使用 raw_decode 逐个解析 JSON 对象
    decoder = json.JSONDecoder()
    pos = 0
    total_len = len(content)

    while pos < total_len:
        # 跳过空白字符 (空格、换行等)
        while pos < total_len and content[pos].isspace():
            pos += 1
        
        if pos >= total_len:
            break

        try:
            # raw_decode 返回 (解析出的对象, 解析结束后的新位置)
            item, end_pos = decoder.raw_decode(content, idx=pos)
            pos = end_pos # 更新位置指针
            
            # ================= 评估逻辑开始 =================
            gt = item.get("ground_truth")
            pred = item.get("prediction")

            # 跳过没有标签的数据 或 预测为空的数据
            if gt is None or pred is None:
                continue

            stats["total"] += 1

            # 确保 pred 是布尔值 (防止模型输出 "true"/"false" 字符串)
            if isinstance(pred, str):
                pred = True if pred.lower() == 'true' else False

            if gt == pred:
                stats["correct"] += 1
                if gt is True: stats["tp"] += 1
                else: stats["tn"] += 1
            else:
                # 记录错误类型
                if pred is True: 
                    stats["fp"] += 1
                    item["error_type"] = "False Positive (误报)"
                else: 
                    stats["fn"] += 1
                    item["error_type"] = "False Negative (漏报)"
                
                error_cases.append(item)
            # ================= 评估逻辑结束 =================

        except json.JSONDecodeError:
            # 如果解析失败，说明这里有非法字符，跳过一个字符尝试继续
            # print(f"⚠️ 解析警告: 在位置 {pos} 跳过非法字符")
            pos += 1

    # --- 计算指标 ---
    if stats["total"] == 0:
        print("❌ 未找到有效数据，请检查 JSON 字段名是否匹配 (ground_truth / sarcasm_prediction)")
        return

    acc = stats["correct"] / stats["total"]
    # 防止除以零
    precision = stats["tp"] / (stats["tp"] + stats["fp"]) if (stats["tp"] + stats["fp"]) > 0 else 0
    recall = stats["tp"] / (stats["tp"] + stats["fn"]) if (stats["tp"] + stats["fn"]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- 打印报告 ---
    print(f"\n{'='*30}")
    print(f"📊 评估结果 (样本数: {stats['total']})")
    print(f"{'='*30}")
    print(f"✅ Accuracy (准确率):  {acc:.2%}")
    print(f"🎯 Precision (查准率): {precision:.2%}")
    print(f"🔎 Recall    (查全率): {recall:.2%}")
    print(f"⚖️  F1-Score:         {f1:.2%}")
    print(f"{'-'*30}")
    print(f"详细统计:")
    print(f"  TP (讽刺且预测对): {stats['tp']}")
    print(f"  TN (正常且预测对): {stats['tn']}")
    print(f"  FP (误判为讽刺):   {stats['fp']} (模型太敏感)")
    print(f"  FN (漏判讽刺):     {stats['fn']} (模型没识别出)")
    print(f"{'='*30}")

if __name__ == "__main__":
    evaluate_and_extract_errors()

📂 正在读取并修复文件格式: ./results_ablation/remove_phase1_au2.jsonl ...

📊 评估结果 (样本数: 690)
✅ Accuracy (准确率):  68.41%
🎯 Precision (查准率): 65.99%
🔎 Recall    (查全率): 75.94%
⚖️  F1-Score:         70.62%
------------------------------
详细统计:
  TP (讽刺且预测对): 262
  TN (正常且预测对): 210
  FP (误判为讽刺):   135 (模型太敏感)
  FN (漏判讽刺):     83 (模型没识别出)
